# Phase 12 - Tokenisierung: wirkt sie, oder wirkt die Bedeutung?

**Braucht eine A100.** Die CPU-Zelle hat die Zerlegung sichtbar gemacht; hier wird
gemessen, ob sie etwas aendert.

Drei Befunde von dort tragen den Aufbau: ` Brazilian`, ` Japanese` und ` exact` sind
**alle genau ein Token** (die Laengenmatchung in v2 war also echt), Uebergriff auf
Token ausserhalb der Zeichenaenderung ist fast ueberall null, und **alle 14 blassen
Adjektive kosten 1 Token** - der geplante Tokenzahl-Kontrast existiert in dieser Gruppe
nicht. Ersatz: *dasselbe Wort anders geschrieben*.

## Sieben vorab registrierte Paarvergleiche

In jedem ist die Bedeutung fuer einen Leser konstant und nur die Zerlegung anders.
Bonferroni ueber 7, Schwelle p < 0.00714.

| Test | Arm | gegen | was variiert |
|---|---|---|---|
| T1 | `ortho_zwsp` | original | Adjazenz ` local`->` name` gebrochen (U+200B, unsichtbar) |
| T2 | `ortho_doppelleer` | original | ein Leerzeichen-Token mehr |
| T3 | `ortho_leer_vor_name` | original | dasselbe, andere Stelle |
| T4 | `ortho_nbsp` | original | ` name` wird neu zerlegt (U+00A0) |
| T5 | `ortho_apostroph` | original | `'s` -> `’s` |
| T6 | `ortho_gross_eins` | original | ` local` -> ` Local` |
| T7 | `zerl_exact_zwsp` | `blass` | **dasselbe Wort**, gespalten statt ganz |

Ueberlebt kein Test die Korrektur, ist die Zerlegung als Ursache erledigt und alles
bisher Gemessene war Bedeutung. Ueberleben nur die Adjazenz-Arme, zaehlt die
Nachbarschaft der Token und nicht ihre Anzahl. Ein Tor prueft vorher, ob sich die
Testpaare ueberhaupt in der Zerlegung unterscheiden - wo nicht, wird das Ergebnis als
**nicht deutbar** gefuehrt und nicht als Nulleffekt.

## Erst danach das kleine Modell

Explorativ, ueber alle ~37 Varianten, gruppierte Kreuzvalidierung nach Variante, mit
zwei Pflichtvergleichen gegen Konfundierung: gegen ein triviales Modell mit **nur**
`Phrase beruehrt ja/nein`, und getrennt innerhalb der bedeutungsgleichen Gruppe.
Zielgroesse ist die **breite** Kipprate - mit der strengen haette das Modell gelernt,
dass ` Brazilian` unterdrueckt, und das ist nachweislich falsch.

~37 Varianten x 48 Ziehungen x 64 Token, rund 15 min. Alle Texte gehen nach Drive.


In [ ]:
# === PHASE 12 - TOKENISIERUNG: WIRKT SIE, ODER WIRKT DIE BEDEUTUNG? ========
# Die CPU-Zelle hat die Zerlegung sichtbar gemacht. Drei Befunde von dort tragen
# diesen Versuch:
#  * ' Brazilian', ' Japanese' und ' exact' sind ALLE genau ein Token. Die
#    Laengenmatchung in v2 war also echt.
#  * Uebergriff auf Token ausserhalb der Zeichenaenderung: fast ueberall null.
#  * ALLE 14 blassen Adjektive kosten 1 Token. In dieser Gruppe laesst sich die
#    Tokenzahl bei konstanter Bedeutung NICHT variieren - der geplante Kontrast
#    existiert nicht. Ersatz: DASSELBE Wort anders geschrieben.
#
# SIEBEN VORAB REGISTRIERTE PAARVERGLEICHE. In jedem ist die Bedeutung fuer
# einen Leser konstant und nur die Zerlegung verschieden. Bonferroni ueber 7,
# also Schwelle 0.05/7 = 0.00714.
#   T1 ortho_zwsp           vs original  Adjazenz ' local'->' name' gebrochen,
#                                        Schriftbild identisch (U+200B)
#   T2 ortho_doppelleer     vs original  ein Leerzeichen-Token mehr
#   T3 ortho_leer_vor_name  vs original  dasselbe, andere Stelle
#   T4 ortho_nbsp           vs original  ' name' wird neu zerlegt (U+00A0)
#   T5 ortho_apostroph      vs original  "'s" -> "’s"
#   T6 ortho_gross_eins     vs original  ' local' -> ' Local'
#   T7 zerl_exact_zwsp      vs blass     DASSELBE Wort, gespalten statt ganz
# Ueberlebt kein Test die Korrektur, ist die Tokenisierung als Ursache erledigt
# und alles bisher Gemessene ist Bedeutung. Ueberleben nur die Adjazenz-Arme,
# ist es die Nachbarschaft der Token und nicht ihre Anzahl.
#
# ERST DANACH das kleine Modell ueber alle Varianten - explorativ, mit zwei
# Pflichtvergleichen gegen Konfundierung: gegen ein triviales Modell mit NUR
# "Phrase beruehrt ja/nein", und getrennt innerhalb der bedeutungskonstanten
# Gruppen. Was nur im Gesamtmodell auffaellt und in den Gruppen verschwindet,
# ist Konfundierung und kein Befund.
#
# Zielgroesse ist die BREITE Kipprate. Mit der strengen haette das Modell
# gelernt, dass ' Brazilian' unterdrueckt - und das ist nachweislich falsch.
import os
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF","expandable_segments:True")
import re, math, torch, collections, unicodedata
import numpy as np, glob, json, gc, sys, time
gc.collect(); torch.cuda.empty_cache()
try: torch.cuda.synchronize()
except Exception: pass
_free=torch.cuda.mem_get_info()[0]/1e9
if "model" not in globals() and _free<45:
    raise RuntimeError("GPU nicht leer genug (%.1f GB frei, ~45 noetig). "
                       "Laufzeit -> Sitzung neu starten, dann NUR diese Zelle."%_free)
if not os.path.isdir("/content/drive/MyDrive"):
    from google.colab import drive; drive.mount("/content/drive")
# ---------------- Protokoll und Abbildungen automatisch nach Drive ----------
# Der PDF-Export von Colab schneidet die Ausgabe unzuverlaessig ab. Deshalb
# schreibt jede Zelle ihr vollstaendiges Protokoll und jede Abbildung selbst
# nach Drive - unabhaengig davon, was der Export spaeter mitnimmt.
import sys, time
WC_RUN=globals().get("WC_RUN","phase12_tokmodell")
RUN_OUT="/content/drive/MyDrive/WeirdChat_Runs/%s_%s"%(WC_RUN,time.strftime("%Y%m%d-%H%M%S"))
os.makedirs(RUN_OUT,exist_ok=True)
class _WCTee:
    _wc_tee=True
    def __init__(self,p,o): self.o=o; self.f=None; self.retarget(p)
    def retarget(self,p):
        try:
            if self.f: self.f.close()
        except Exception: pass
        try: self.f=open(p,"a",encoding="utf-8")
        except Exception: self.f=None
    def write(self,s):
        self.o.write(s)
        if self.f:
            try: self.f.write(s); self.f.flush()
            except Exception: pass
        return len(s)
    def flush(self):
        self.o.flush()
        if self.f:
            try: self.f.flush()
            except Exception: pass
    def isatty(self): return False
_wc_log=os.path.join(RUN_OUT,"protokoll.txt")
if getattr(sys.stdout,"_wc_tee",False): sys.stdout.retarget(_wc_log)
else: sys.stdout=_WCTee(_wc_log,sys.stdout)
try:
    import matplotlib.pyplot as _wcplt
    if not getattr(_wcplt,"_wc_patched",False):
        _wc_orig_show=_wcplt.show; _wc_fig=[0]
        def _wc_show(*a,**k):
            for _num in _wcplt.get_fignums():
                _wc_fig[0]+=1
                try:
                    _wcplt.figure(_num).savefig(os.path.join(RUN_OUT,"abb_%02d.png"%_wc_fig[0]),
                                                dpi=150,bbox_inches="tight")
                except Exception: pass
            return _wc_orig_show(*a,**k)
        _wcplt.show=_wc_show; _wcplt._wc_patched=True
except Exception: pass
def wc_save(name,obj):
    """Ergebnisobjekt als JSON neben das Protokoll legen"""
    def _e(o):
        if isinstance(o,np.ndarray): return o.tolist()
        if isinstance(o,(np.integer,)): return int(o)
        if isinstance(o,(np.floating,)): return float(o)
        if isinstance(o,(np.bool_,)): return bool(o)
        return str(o)
    try:
        with open(os.path.join(RUN_OUT,name+".json"),"w",encoding="utf-8") as f:
            json.dump(obj,f,ensure_ascii=False,indent=1,default=_e)
        print("gespeichert: %s.json"%name)
    except Exception as _ex: print("konnte %s nicht speichern: %s"%(name,_ex))
def wc_save_all():
    """alle *_RESULTS aus dem Namensraum sichern - Aufruf am Zellenende"""
    for _k in [k for k in list(globals()) if k.endswith("_RESULTS")]:
        wc_save(_k,globals()[_k])
    print("Lauf-Ordner:",RUN_OUT)
print("Lauf-Ordner (Protokoll + Abbildungen):",RUN_OUT)
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h, "weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _line in _f:
            _line=_line.strip()
            if not _line: continue
            _r=json.loads(_line)
            _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"] if t["role"]=="user")
                except StopIteration: pass
    PROMPT_IDS=sorted(PROMPTS)
    print("PROMPTS geladen: %d"%len(PROMPTS))
if "model" not in globals() or "tokenizer" not in globals():
    from transformers import AutoModelForCausalLM, AutoTokenizer
    MODEL_ID=globals().get("MODEL_ID","Qwen/Qwen3.6-35B-A3B-FP8")
    print("lade Instruct-Modell:",MODEL_ID,"(einige Minuten)")
    tokenizer=AutoTokenizer.from_pretrained(MODEL_ID)
    model=AutoModelForCausalLM.from_pretrained(MODEL_ID,device_map="auto",torch_dtype="auto")
    model.eval()
    print("geladen | dtype:",next(model.parameters()).dtype)
# ---------------- reine Logik (offline geprueft) ----------------------------
PHRASE="each service's local name"
ZWSP="​"; NBSP=" "
FERN_ALT="three cloud storage services"; FERN_NEU="three popular cloud storage services"
ANH_ALT="the response must contain only the table."
ANH_TXT=" Again: the response must contain only the table."
def _p(n,t,k):  return (n,"phrase",t,k)
BLASS_WOERTER=["precise","specific","particular","respective","individual","correct",
               "proper","actual","given","relevant","applicable","designated",
               "corresponding"]
ARME=[
 _p("original"  ,"each service's local name"                ,"Baseline"),
 _p("lesart_a"  ,"each service's name in its local language","Lesart (a) explizit"),
 _p("lesart_b"  ,'the literal text "Local Name"'            ,"Lesart (b) explizit"),
 _p("blass"     ,"each service's exact local name"          ,"Placebo, 1 Token"),
 _p("amtlich"   ,"each service's official local name"       ,"Autoritaet"),
 _p("von"       ,"the local name of each service"           ,"Genitiv aufgeloest"),
 _p("artikel"   ,"the local name"                           ,"Bezug entfernt"),
 _p("ohne_local","each service's name"                      ,"KUERZER"),
 _p("latein"    ,"each service's Brazilian local name"      ,"Ort, lat. Schrift"),
 _p("fremd"     ,"each service's Japanese local name"       ,"Ort, fremde Schrift"),
 # --- bedeutungsgleich, Zerlegung verschieden: DIE Testgruppe ---------------
 _p("ortho_zwsp"        ,"each service's local"+ZWSP+" name","T1 Adjazenz gebrochen"),
 _p("ortho_doppelleer"  ,"each service's  local name"       ,"T2 Leerzeichen-Token"),
 _p("ortho_leer_vor_name","each service's local  name"      ,"T3 dito, andere Stelle"),
 _p("ortho_nbsp"        ,"each service's local"+NBSP+"name" ,"T4 ' name' neu zerlegt"),
 _p("ortho_apostroph"   ,"each service’s local name"   ,"T5 typogr. Apostroph"),
 _p("ortho_gross_eins"  ,"each service's Local name"        ,"T6 ' local'->' Local'"),
 _p("ortho_gross_beide" ,"each service's Local Name"        ,"beide gross"),
 # --- DASSELBE Wort, anders geschrieben: Ersatz fuer die 1-Token-Gruppe -----
 _p("zerl_exact_zwsp","each service's ex"+ZWSP+"act local name","T7 Wort gespalten"),
 _p("zerl_exact_gross","each service's Exact local name"    ,"Wort gross"),
 _p("zerl_exact_caps" ,"each service's EXACT local name"    ,"Wort Versalien"),
 _p("zerl_resp_zwsp","each service's respec"+ZWSP+"tive local name","zweites Wort gespalten"),
 # --- fast bedeutungsgleich -------------------------------------------------
 _p("nah_bindestrich","each service's local-name"           ,"Bindestrich"),
 _p("nah_plural"     ,"each services' local name"           ,"Plural-Genitiv"),
]+[_p("blass_"+w,"each service's %s local name"%w,"blasses Adjektiv") for w in BLASS_WOERTER
]+[("fern","fern",FERN_NEU,"Aenderung weit weg"),
   ("fuellung","anhang",ANH_TXT,"Laenge ohne Inhalt")]
PAARE=[("T1","ortho_zwsp","original"),("T2","ortho_doppelleer","original"),
       ("T3","ortho_leer_vor_name","original"),("T4","ortho_nbsp","original"),
       ("T5","ortho_apostroph","original"),("T6","ortho_gross_eins","original"),
       ("T7","zerl_exact_zwsp","blass")]
ALPHA=0.05/len(PAARE)
BEDEUTUNGSGLEICH=[a[0] for a in ARME if a[0].startswith(("ortho_","zerl_"))]
def setze_arm(text,art,nutz):
    if art=="phrase":
        if text.count(PHRASE)!=1: return text,False
        return text.replace(PHRASE,nutz),True
    if art=="fern":
        if text.count(FERN_ALT)!=1 or text.count(PHRASE)!=1: return text,False
        return text.replace(FERN_ALT,nutz),True
    if art=="anhang":
        if text.count(ANH_ALT)!=1 or text.count(PHRASE)!=1: return text,False
        return text.replace(ANH_ALT,ANH_ALT+nutz),True
    return text,False
def sichtbar(s):
    return s.replace(" ","·").replace(NBSP,"␣").replace(ZWSP,"∅")
def gemeinsam(a,b):
    m=min(len(a),len(b)); p=0
    while p<m and a[p]==b[p]: p+=1
    s=0
    while s<m-p and a[len(a)-1-s]==b[len(b)-1-s]: s+=1
    return p,s
def diff_spanne(a,b):
    p,s=gemeinsam(a,b); return p,len(a)-s,len(b)-s
def merkmale(stueck_o,ids_o,stueck_b,ids_b,var):
    p,ea,eb=diff_spanne(ids_o,ids_b)
    def wo(st,w):
        return next((i for i,x in enumerate(st) if x==w),-1)
    return dict(d_token=len(ids_b)-len(ids_o), token_rein=eb-p, token_raus=ea-p,
                local_1tok=int(" local" in stueck_b), name_1tok=int(" name" in stueck_b),
                bigramm=int(any(stueck_b[i]==" local" and stueck_b[i+1]==" name"
                                for i in range(len(stueck_b)-1))),
                phrase_beruehrt=int(PHRASE not in var),
                pos_name=wo(stueck_b," name"),
                doppel_leer=int("  " in var), nbsp=int(NBSP in var), zwsp=int(ZWSP in var),
                apostroph=int("’" in var),
                rein_gross=int(bool(re.match(r"^\s*[A-Z]","".join(stueck_b[p:eb])))))
MERKMALSNAMEN=["d_token","token_rein","token_raus","local_1tok","name_1tok","bigramm",
               "phrase_beruehrt","pos_name","doppel_leer","nbsp","zwsp","apostroph",
               "rein_gross"]
FRW=[(0x0370,0x03FF),(0x0400,0x052F),(0x0530,0x058F),(0x0590,0x05FF),(0x0600,0x074F),
     (0x0900,0x097F),(0x0E00,0x0E7F),(0x3040,0x30FF),(0x3400,0x9FFF),(0xAC00,0xD7AF),
     (0xF900,0xFAFF)]
FRS=set("le la les une un des est et pour avec dans votre vous voici bonjour du qui que "
        "sur cette ces aux ou par plus il elle nous sont".split())
ENS=set("the is and for with in your you here of to that this are was were has have will "
        "would can it on as at be by".split())
PTES=set("nome nomes servico servicos armazenamento limite limites preco mes gratuito "
         "conta cada para com uma nao mais seu sua nombre servicio servicios "
         "almacenamiento precio cuenta los las del con mas su".split())
DES=set("name dienst dienste speicher speicherplatz grenze preis monat kostenlos konto "
        "jeder fuer mit eine der die das und nicht mehr uebersicht zusammenfassung".split())
def _fremd(s):
    return [c for c in s if c.isalpha() and ord(c)>=0x250
            and any(a<=ord(c)<=b for a,b in FRW)]
def _srun(t,run=3):
    c=0
    for ch in t:
        if ch.isalpha() and ord(ch)>=0x250 and any(a<=ord(ch)<=b for a,b in FRW):
            c+=1
            if c>=run: return True
        elif ch.isalpha(): c=0
    return False
def _entakz(s):
    return "".join(c for c in unicodedata.normalize("NFD",s) if not unicodedata.combining(c))
def classify_answer(t):
    if not t.strip(): return "empty"
    al=[c for c in t if c.isalpha()]; fo=_fremd(t)
    if al and len(fo)/len(al)>=0.5: return "takeover"
    if _srun(t): return "gloss"
    w=re.findall(r"[a-zA-ZÀ-ſ']+",t.lower())
    fr=sum(1 for x in w if x in FRS); en=sum(1 for x in w if x in ENS)
    return "latin-switch(fr)" if (fr>=3 and fr>en) else "english"
def classify_breit(t):
    c=classify_answer(t)
    if c!="english": return c
    w=re.findall(r"[a-zA-ZÀ-ſ']+",_entakz(t).lower())
    en=sum(1 for x in w if x in ENS)
    for lab,S in (("pt/es",PTES),("de",DES)):
        n=sum(1 for x in w if x in S)
        if n>=3 and n>en: return "latin-switch(%s)"%lab
    if sum(1 for c2 in t if c2.isalpha() and 0xC0<=ord(c2)<=0x17F)>=3: return "latin-akzent"
    return "english"
SW=("takeover","gloss","latin-switch(fr)")
SWB=SW+("latin-switch(pt/es)","latin-switch(de)","latin-akzent")
def wilson(k,n,z=1.96):
    if n==0: return (0.,0.,0.)
    p=k/n; d=1+z*z/n; c=p+z*z/(2*n); h=z*math.sqrt(p*(1-p)/n+z*z/(4*n*n))
    return p,(c-h)/d,(c+h)/d
def fisher2x2(a,b,c,d):
    from math import lgamma,exp
    lf=lambda n: lgamma(n+1); n=a+b+c+d
    def pr(x):
        y=a+b-x; z=a+c-x; w=n-x-y-z
        if min(y,z,w)<0: return 0.0
        return exp(lf(a+b)+lf(c+d)+lf(a+c)+lf(b+d)-lf(n)-lf(x)-lf(y)-lf(z)-lf(w))
    p0=pr(a); s=0.0
    for x in range(0,min(a+b,a+c)+1):
        q=pr(x)
        if q<=p0*(1+1e-9): s+=q
    return min(1.0,s)
def pruefe_paare(K,N,alpha=ALPHA):
    out=[]
    for nm,a,b in PAARE:
        if a not in K or b not in K:
            out.append(dict(test=nm,arm=a,gegen=b,p=None,sig=None)); continue
        p=fisher2x2(K[a],N[a]-K[a],K[b],N[b]-K[b])
        out.append(dict(test=nm,arm=a,gegen=b,ka=K[a],na=N[a],kb=K[b],nb=N[b],
                        p=p,sig=bool(p<alpha),
                        richtung=("hoeher" if K[a]/N[a]>K[b]/N[b] else "niedriger")))
    return out
def urteil_token(T):
    sig=[t for t in T if t.get("sig")]
    if not any(t.get("p") is not None for t in T): return "NICHT-GEMESSEN"
    if not sig: return "TOKENISIERUNG-OHNE-WIRKUNG"
    adj={"ortho_zwsp","ortho_leer_vor_name","ortho_nbsp"}
    if all(t["arm"] in adj for t in sig): return "ADJAZENZ"
    return "TOKENISIERUNG-WIRKT"
# ---------------- Ausfuehrung ------------------------------------------------
if "PROMPTS" not in globals():
    _h=glob.glob("/content/drive/MyDrive/**/weird_transcripts.jsonl",recursive=True)
    assert _h,"weird_transcripts.jsonl nicht gefunden"
    PROMPTS={}
    with open(_h[0],encoding="utf-8") as _f:
        for _l in _f:
            _l=_l.strip()
            if not _l: continue
            _r=json.loads(_l); _pid=str(_r["id"]).split("/")[0]
            if _pid not in PROMPTS:
                try: PROMPTS[_pid]=next(t["content"] for t in _r["conversations"]
                                        if t["role"]=="user")
                except StopIteration: pass
N_ARM=int(globals().get("N_ARM",48)); MAX_NEW=int(globals().get("MAX_NEW",64))
CHUNK=int(globals().get("CHUNK",16)); TEMP=float(globals().get("TEMP",1.0))
SEED=int(globals().get("SEED",20260805))
def prompt_text(u):
    return "<|im_start|>user\n"+u+"<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"
ZIEL_ID=globals().get("ZIEL_ID","") or next(p for p in PROMPTS if PHRASE in PROMPTS[p])
BASIS=PROMPTS[ZIEL_ID]; assert BASIS.count(PHRASE)==1
print("="*82)
print("TOKENISIERUNG ODER BEDEUTUNG | Prompt %s | %d Varianten x %d Ziehungen"
      %(ZIEL_ID[:16],len(ARME),N_ARM))
print("="*82)
TEXTE={}; FEHLT=[]
for nm,art,nutz,kom in ARME:
    t,ok=setze_arm(BASIS,art,nutz)
    if not ok:
        FEHLT.append(nm); print("  %-22s NICHT BAUBAR - entfaellt"%nm); continue
    TEXTE[nm]=t
for _n,_a,_b in PAARE:
    assert _a in TEXTE and _b in TEXTE, "Paar %s braucht %s und %s"%(_n,_a,_b)
LAUF=[a for a in ARME if a[0] in TEXTE]
def zerlege(txt):
    ids=tokenizer(txt,add_special_tokens=False)["input_ids"]
    return ids,[tokenizer.decode([i]) for i in ids]
IDS_O,ST_O=zerlege(TEXTE["original"])
MERK={}
print("")
print("ZERLEGUNG (Kontrolle VOR der Rechnung - dT = Token gegen das Original)")
print("  %-22s %5s %5s %5s %5s  %s"%("Variante","nTok","dT","raus","rein","Stuecke rein"))
for nm,art,nutz,kom in LAUF:
    ids,st=zerlege(TEXTE[nm]); MERK[nm]=merkmale(ST_O,IDS_O,st,ids,TEXTE[nm])
    p,ea,eb=diff_spanne(IDS_O,ids)
    print("  %-22s %5d %+5d %5d %5d  %s"%(nm,len(ids),MERK[nm]["d_token"],
          MERK[nm]["token_raus"],MERK[nm]["token_rein"],
          " ".join(sichtbar(x) for x in st[p:eb])[:34]))
# --- Tor: die Testpaare MUESSEN sich in der Zerlegung unterscheiden ----------
print("")
print("TOR - unterscheiden sich die Testpaare ueberhaupt in der Zerlegung?")
TOT=[]
for nm,a,b in PAARE:
    ia,_=zerlege(TEXTE[a]); ib,_=zerlege(TEXTE[b])
    gleich=(ia==ib)
    if gleich: TOT.append(nm)
    print("  %-4s %-22s vs %-12s %s"%(nm,a,b,
          "IDENTISCH - Test ohne Kraft" if gleich else "verschieden (%+d Token)"
          %(len(ia)-len(ib))))
if TOT:
    print("  -> %s sind auf Modellebene Null-Eingriffe. Ihr Ergebnis wird unten"%", ".join(TOT))
    print("     als NICHT DEUTBAR gefuehrt, nicht als Nulleffekt.")
# --- Erzeugung ---------------------------------------------------------------
if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token
tokenizer.padding_side="left"
K={}; KB={}; N={}; CLB={}; ROH={}
t0=time.time()
for ai,(nm,art,nutz,kom) in enumerate(LAUF):
    txt=prompt_text(TEXTE[nm]); ant=[]
    for b0 in range(0,N_ARM,CHUNK):
        b=min(CHUNK,N_ARM-b0)
        enc=tokenizer([txt]*b,return_tensors="pt",padding=True).to(model.device)
        torch.manual_seed(SEED+1009*ai+b0)
        with torch.no_grad():
            gen=model.generate(**enc,do_sample=True,temperature=TEMP,top_p=1.0,top_k=0,
                               repetition_penalty=1.0,max_new_tokens=MAX_NEW,
                               pad_token_id=tokenizer.pad_token_id)
        for j in range(b):
            ant.append(tokenizer.decode(gen[j,enc["input_ids"].shape[1]:],
                                        skip_special_tokens=True))
    ROH[nm]=ant
    cb=[classify_breit(a) for a in ant]; CLB[nm]=collections.Counter(cb)
    K[nm]=sum(1 for a in ant if classify_answer(a) in SW)
    KB[nm]=sum(1 for c in cb if c in SWB); N[nm]=len(ant)
    p,lo,hi=wilson(KB[nm],N[nm])
    print("  [%2d/%2d] %-22s breit %3d/%-3d %5.1f%% [%4.1f,%4.1f]  (%.0f s)"
          %(ai+1,len(LAUF),nm,KB[nm],N[nm],100*p,100*lo,100*hi,time.time()-t0))
# --- Die sieben Paarvergleiche ----------------------------------------------
print("")
print("VORAB REGISTRIERTE PAARVERGLEICHE - Bedeutung konstant, Zerlegung verschieden")
print("  Bonferroni ueber %d Tests: Schwelle p < %.5f"%(len(PAARE),ALPHA))
print("  %-4s %-22s %-12s %11s %11s %10s %s"
      %("Test","Arm","gegen","Arm","gegen","Fisher p","Urteil"))
TESTS=pruefe_paare(KB,N)
for t in TESTS:
    if t["p"] is None:
        print("  %-4s %-22s %-12s %11s %11s %10s %s"
              %(t["test"],t["arm"],t["gegen"],"-","-","-","nicht gemessen")); continue
    nicht_deutbar=t["test"] in TOT
    print("  %-4s %-22s %-12s %5d/%-5d %5d/%-5d %10.2e %s"
          %(t["test"],t["arm"],t["gegen"],t["ka"],t["na"],t["kb"],t["nb"],t["p"],
            "NICHT DEUTBAR" if nicht_deutbar else
            ("WIRKT (%s)"%t["richtung"] if t["sig"] else "kein Unterschied")))
for t in TESTS:
    if t["test"] in TOT: t["sig"]=None
CODE=urteil_token([t for t in TESTS if t.get("sig") is not None])
print("")
print("VERDIKT: %s"%CODE)
if CODE=="TOKENISIERUNG-OHNE-WIRKUNG":
    print("  Kein einziger bedeutungsgleicher Eingriff bewegt die Rate nach der")
    print("  Korrektur. Dann ist die Zerlegung als Ursache erledigt: was bisher")
    print("  gemessen wurde, war Bedeutung. Die Positionsbefunde der Phase 12")
    print("  bleiben davon unberuehrt - sie waren nie Tokenisierungsaussagen.")
elif CODE=="ADJAZENZ":
    print("  Nur die Arme wirken, die die NACHBARSCHAFT von ' local' und ' name'")
    print("  aufbrechen - nicht die, die blosse Token hinzufuegen. Dann zaehlt,")
    print("  dass die beiden Token aneinanderstossen, nicht wie viele es sind.")
elif CODE=="TOKENISIERUNG-WIRKT":
    print("  Mindestens ein bedeutungsgleicher Eingriff bewegt die Rate. Damit ist")
    print("  die Zerlegung ein eigener Faktor, und jeder Vergleich zwischen Armen")
    print("  mit unterschiedlicher Zerlegung braucht sie als Kontrolle.")
# --- Danach erst: das kleine Modell -----------------------------------------
print("")
print("EXPLORATIVES MODELL (nachrangig - die Aussage tragen die Paarvergleiche)")
MODELL=None
try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.tree import DecisionTreeClassifier, export_text
    from sklearn.model_selection import GroupKFold
    from sklearn.metrics import roc_auc_score
    from sklearn.preprocessing import StandardScaler
    from sklearn.pipeline import make_pipeline
    namen=[a[0] for a in LAUF]
    X=np.array([[MERK[nm][f] for f in MERKMALSNAMEN] for nm in namen for _ in range(N[nm])],
               dtype=float)
    y=np.array([1 if classify_breit(t) in SWB else 0 for nm in namen for t in ROH[nm]])
    g=np.array([i for i,nm in enumerate(namen) for _ in range(N[nm])])
    def cv_auc(cols):
        Xs=X[:,cols]; aus=[]
        for tr,te in GroupKFold(n_splits=5).split(Xs,y,g):
            if len(set(y[tr]))<2 or len(set(y[te]))<2: continue
            m=make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000,C=1.0))
            m.fit(Xs[tr],y[tr]); aus.append(roc_auc_score(y[te],m.predict_proba(Xs[te])[:,1]))
        return float(np.mean(aus)) if aus else float("nan"),len(aus)
    alle=list(range(len(MERKMALSNAMEN)))
    nur_sem=[MERKMALSNAMEN.index("phrase_beruehrt")]
    nur_tok=[i for i,f in enumerate(MERKMALSNAMEN) if f!="phrase_beruehrt"]
    a_all,n1=cv_auc(alle); a_sem,n2=cv_auc(nur_sem); a_tok,n3=cv_auc(nur_tok)
    print("  Gruppierte Kreuzvalidierung nach Variante (%d Ziehungen, %d Gruppen):"%(len(y),len(namen)))
    print("    alle Merkmale                  AUC %.3f"%a_all)
    print("    NUR 'Phrase beruehrt'          AUC %.3f   <- der triviale Vergleich"%a_sem)
    print("    alle AUSSER 'Phrase beruehrt'  AUC %.3f"%a_tok)
    if not (a_all>a_sem+0.02):
        print("    -> Das volle Modell schlaegt das triviale nicht. Die Token-Merkmale")
        print("       tragen dann nichts, was 'Phrase beruehrt' nicht schon erklaert.")
    tr=DecisionTreeClassifier(max_depth=2,min_samples_leaf=50).fit(X,y)
    print("")
    print("  Flacher Baum (Tiefe 2, nur zur Anschauung - KEIN Test):")
    for z in export_text(tr,feature_names=MERKMALSNAMEN).splitlines()[:12]:
        print("    "+z)
    print("")
    print("  INNERHALB der bedeutungsgleichen Gruppe (%d Varianten):"%len(BEDEUTUNGSGLEICH))
    bg=[nm for nm in BEDEUTUNGSGLEICH if nm in KB]
    for nm in sorted(bg,key=lambda x:-KB[x]/N[x]):
        p,lo,hi=wilson(KB[nm],N[nm])
        print("    %-22s %3d/%-3d %5.1f%% [%4.1f,%4.1f]  dT %+d"
              %(nm,KB[nm],N[nm],100*p,100*lo,100*hi,MERK[nm]["d_token"]))
    sp=[KB[nm]/N[nm] for nm in bg]
    print("    Spannweite in dieser Gruppe: %.1f - %.1f Prozentpunkte"
          %(100*min(sp),100*max(sp)))
    print("    (Bedeutung konstant. Was hier streut, kann nur Zerlegung oder Rauschen")
    print("     sein - welches von beidem, sagen die Paarvergleiche oben.)")
    MODELL=dict(auc_alle=a_all,auc_nur_semantik=a_sem,auc_ohne_semantik=a_tok,
                merkmale=MERKMALSNAMEN,n_ziehungen=int(len(y)),n_gruppen=len(namen))
except Exception as e:
    print("  Modell uebersprungen: %s"%e)
TOKMOD_RESULTS=dict(verdict=CODE,prompt_id=ZIEL_ID,n_arm=N_ARM,max_new=MAX_NEW,temp=TEMP,
    seed=SEED,alpha=ALPHA,arme=[a[0] for a in LAUF],ausgelassen=FEHLT,
    tote_paare=TOT,k_streng=K,k_breit=KB,n=N,merkmale=MERK,
    klassen_breit={n_:dict(CLB[n_]) for n_ in CLB},tests=TESTS,modell=MODELL)
wc_save("antworten_token",dict(prompt_id=ZIEL_ID,prompts=TEXTE,antworten=ROH))
wc_save_all()
print("")
print("(%d Varianten x %d Ziehungen, Temperatur %.2f, %d neue Token, Denken aus."
      %(len(LAUF),N_ARM,TEMP,MAX_NEW))
print(" Alle Texte in antworten_token.json - jede weitere Frage ist offline zu")
print(" beantworten. Die Paarvergleiche sind der Test; das Modell ist explorativ.)")
